# FYP Thesis — Colab Setup

Run cells top-to-bottom once per Colab session.  
All data and checkpoints persist on Google Drive; only `/content/data/processed` is local (fast I/O scratch).

**Prerequisites**
- Google Drive mounted with folder `MyDrive/FYP_Thesis/data/` already created (or will be created below)
- Colab secret `KAGGLE_API_TOKEN` set in *Settings → Secrets* (JSON string: `{"username":"...","key":"..."}`)

## 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('Drive mounted.')

## 2 — Clone or pull the repo

In [ ]:
import os

REPO_URL = "https://github.com/Coolboy-786/University-Final-year-project.git"
REPO_DIR = "/content/skin-disease-cls"

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
    print('Repo updated.')
else:
    !git clone {REPO_URL} {REPO_DIR}
    print('Repo cloned.')

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

## 3 — Install dependencies

In [ ]:
!pip install -e . --quiet
print('Dependencies installed.')

## 4 — Symlink repo/data → Drive

In [ ]:
import os
import shutil
from pathlib import Path

DRIVE_DATA = Path('/content/drive/MyDrive/FYP_Thesis (1)/data')
REPO_DATA  = Path(REPO_DIR) / 'data'

DRIVE_DATA.mkdir(parents=True, exist_ok=True)

if REPO_DATA.is_symlink():
    print(f'Symlink already exists: {REPO_DATA} -> {os.readlink(REPO_DATA)}')
elif REPO_DATA.exists():
    shutil.rmtree(REPO_DATA)
    REPO_DATA.symlink_to(DRIVE_DATA)
    print(f'Symlink created: {REPO_DATA} -> {DRIVE_DATA}')
else:
    REPO_DATA.symlink_to(DRIVE_DATA)
    print(f'Symlink created: {REPO_DATA} -> {DRIVE_DATA}')

## 5 — Set STORAGE_BACKEND=colab

In [ ]:
import os
os.environ['STORAGE_BACKEND'] = 'colab'

from src.utils.paths import get_data_root
print(f'Data root: {get_data_root()}')

## 6 — Copy processed data from Drive to /content for fast I/O

In [ ]:
import shutil
from pathlib import Path

DRIVE_PROCESSED = Path('/content/drive/MyDrive/FYP_Thesis (1)/data/processed')
LOCAL_PROCESSED = Path('/content/data/processed')

if DRIVE_PROCESSED.exists():
    if LOCAL_PROCESSED.exists():
        shutil.rmtree(LOCAL_PROCESSED)
    shutil.copytree(DRIVE_PROCESSED, LOCAL_PROCESSED)
    print(f'Copied {DRIVE_PROCESSED} -> {LOCAL_PROCESSED}')
else:
    print('No processed data on Drive yet. Run src.data.merge first, then re-run this cell.')

## 7 — Configure Kaggle credentials from Colab secrets

In [ ]:
from pathlib import Path
from google.colab import userdata

kaggle_json = userdata.get('KAGGLE_API_TOKEN')  # stored as JSON string in Colab secret
kaggle_dir  = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)
kaggle_path = kaggle_dir / 'kaggle.json'
kaggle_path.write_text(kaggle_json)
kaggle_path.chmod(0o600)
print(f'Kaggle credentials written to {kaggle_path}')

## 8 — Verify setup

In [ ]:
from pathlib import Path
from src.utils.paths import get_data_root

root = get_data_root()
checks = {
    'data root':     root,
    'raw dir':       root / 'raw',
    'processed dir': root / 'processed',
    'splits CSV':    root / 'splits.csv',
    'checkpoints':   root / 'checkpoints',
}

for label, path in checks.items():
    status = 'OK' if path.exists() else 'MISSING'
    print(f'  [{status}] {label}: {path}')